# Module 2: Local LLMs with LM Studio
**หลักสูตร:** การพัฒนาเวิร์กโฟลว์ปัญญาประดิษฐ์สำหรับสร้างข้อความและรูปภาพบนเครื่องคอมพิวเตอร์ส่วนบุคคล  
**(Mastering Local AI Stack: Jupyter Notebook × LM Studio × ComfyUI)**

---

## เป้าหมายของบทเรียนนี้
1. เลือกขนาดโมเดล Local LLM (`3B Parameters`, `GGUF Q4_K_M`) และตั้งค่าพารามิเตอร์ให้พอดีกับเครื่องที่มี **VRAM 6GB (RTX 3050)** หรือ **Apple Silicon Unified Memory 16GB**
2. เชื่อมต่อกับ **LM Studio Local Server** (`http://localhost:1234/v1`) ผ่านไลบรารี `openai`
3. ทดลองปรับค่า `system prompt`, `temperature`, และ `max_tokens` เพื่อดูความแตกต่างของคำตอบ
4. **โจทย์ปฏิบัติการ:** เขียนโค้ดอ่านบทความจาก `sample_article.txt` แล้วสั่งให้ Local LLM ดึงประเด็นสำคัญออกมาเป็น **JSON** เพื่อแปลงลงตาราง `pandas.DataFrame`

---
## เตรียมความพร้อมในโปรแกรม LM Studio ก่อนรันโค้ด
1. เปิดโปรแกรม **LM Studio** แล้วโหลดโมเดลขนาด 3B เช่น `Qwen2.5-3B-Instruct-GGUF (Q4_K_M)` หรือ `Llama-3.2-3B-Instruct-GGUF (Q4_K_M)` (ใช้ VRAM ประมาณ 2.2 GB)
2. ตั้งค่า **Context Length** ไว้ที่ `2048` หรือ `4096`
3. คลิกที่แถบ **Developer (Local Server)** ด้านซ้าย แล้วกดปุ่ม **Start Server** ที่พอร์ต `1234`

In [ ]:
from openai import OpenAI

# ตั้งค่า Client ชี้ไปยัง LM Studio Local Server บนเครื่องตัวเอง
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"  # ใส่ข้อความอะไรก็ได้ เพราะเซิร์ฟเวอร์ในเครื่องไม่ตรวจ API Key
)

# ดึงรายชื่อโมเดลที่โหลดค้างไว้ใน LM Studio
try:
    models = client.models.list()
    loaded_model_id = models.data[0].id if models.data else "local-model"
    print(f"เชื่อมต่อ LM Studio สำเร็จ โมเดลที่โหลดอยู่คือ: {loaded_model_id}")
except Exception as e:
    loaded_model_id = "local-model"
    print("ยังเชื่อมต่อ LM Studio ไม่ได้ กรุณาเช็กว่ากด Start Server ที่พอร์ต 1234 แล้วหรือยัง")
    print(f"รายละเอียดข้อผิดพลาด: {e}")

## ส่วนที่ 1: การตั้งค่า System Prompt, Temperature และ Max Tokens

- **System Prompt:** ข้อความกำหนดบทบาทและรูปแบบคำตอบที่เราต้องการจากโมเดล
- **Temperature (`0.0` - `1.0`):** ถ้าตั้งค่าต่ำ (เช่น `0.1` ถึง `0.3`) โมเดลจะเลือกคำที่ตรงไปตรงมาและคาดเดาได้ง่าย เหมาะกับงานสรุปความหรือสั่งทำข้อมูล JSON แต่ถ้าตั้งค่าสูง (เช่น `0.7` ถึง `0.9`) โมเดลจะแต่งประโยคหลากหลายขึ้น
- **Max Tokens:** เพดานจำนวน Token สูงสุดที่ให้โมเดลพิมพ์คำตอบออกมาในรอบนั้น

In [ ]:
response = client.chat.completions.create(
    model=loaded_model_id,
    messages=[
        {
            "role": "system",
            "content": "คุณคือผู้สอนวิชา AI ในระดับมหาวิทยาลัย อธิบายศัพท์เทคนิคให้เข้าใจง่ายและกระชับเป็นภาษาไทย"
        },
        {
            "role": "user",
            "content": "ช่วยอธิบายคำว่า Quantization (เช่น Q4_K_M) ในไฟล์โมเดล GGUF ให้จบภายใน 3 บรรทัดหน่อยครับ"
        }
    ],
    temperature=0.3,
    max_tokens=250
)

print("=== คำตอบจาก Local LLM ===")
print(response.choices[0].message.content)

## ส่วนที่ 2: โจทย์ปฏิบัติการ: สกัดข้อมูลจากบทความเป็น JSON และแสดงผลด้วย Pandas

ในการใช้งานจริง เรามักให้ Local LLM อ่านบทความหรือรายงานภายใน แล้วดึงเฉพาะข้อมูลสำคัญออกมาเป็น **JSON Array** เพื่อเอาไปใส่ตารางหรือเก็บลงฐานข้อมูลต่อ

เราจะอ่านเนื้อหาจากไฟล์ `../assets/sample_article.txt` แล้วสั่งให้โมเดลสกัดประเด็นออกมาเป็นรายการ JSON ที่มีคีย์ดังนี้:
- `topic`: หัวข้อประเด็นสำคัญ
- `category`: หมวดหมู่ (Hardware, Model, Workflow หรือ Benefit)
- `key_takeaway`: สรุปใจความสำคัญสั้นๆ 1 ประโยค

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd

# 1. อ่านไฟล์บทความภาษาไทย
article_text = Path("../assets/sample_article.txt").read_text(encoding="utf-8")

# 2. กำหนด System Prompt เพื่อบังคับให้โมเดลตอบกลับเป็น JSON Array เท่านั้น
extraction_system_prompt = """คุณคือโปรแกรมสกัดข้อมูลจากเอกสาร
ให้อ่านบทความที่ผู้ใช้ส่งมา แล้วดึงประเด็นสำคัญออกมา 4-5 ข้อ
ตอบกลับเป็น JSON Array เท่านั้น ห้ามเขียนข้อความเกริ่นนำหรือคำอธิบายต่อท้าย
โครงสร้าง JSON ที่ต้องใช้:
[
  {
    "topic": "ชื่อหัวข้อ",
    "category": "หมวดหมู่ (Hardware / Model / Workflow / Benefit)",
    "key_takeaway": "สรุปใจความสำคัญภาษาไทย 1 ประโยค"
  }
]"""

extraction_response = client.chat.completions.create(
    model=loaded_model_id,
    messages=[
        {"role": "system", "content": extraction_system_prompt},
        {"role": "user", "content": f"สกัดประเด็นสำคัญจากบทความต่อไปนี้:\n\n{article_text}"}
    ],
    temperature=0.1,
    max_tokens=800
)

raw_output = extraction_response.choices[0].message.content.strip()
print("=== Raw Output จาก Local LLM ===")
print(raw_output)

In [ ]:
# 3. แปลงข้อความ JSON ให้กลายเป็นตาราง Pandas DataFrame
def parse_json_safely(text: str) -> list:
    """ฟังก์ชันตัดเอาเฉพาะส่วน [...] เผื่อกรณีที่โมเดลใส่ Markdown ```json ติดมาด้วย"""
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\[.*\]", cleaned, flags=re.DOTALL)
    target = match.group(0) if match else cleaned
    return json.loads(target)

try:
    extracted_items = parse_json_safely(raw_output)
    df_summary = pd.DataFrame(extracted_items)
    print("แปลงข้อมูล JSON เป็นตาราง Pandas สำเร็จ:")
    display(df_summary)
except Exception as parse_err:
    print(f"ไม่สามารถแปลง JSON ได้: {parse_err}")

---
### ขั้นตอนต่อไป
หลังจากทดลองสั่งให้ Local LLM สกัดข้อมูลเป็น JSON เรียบร้อยแล้ว ในช่วงบ่าย (**Module 3** และ **Module 4**) เราจะไปต่อเวิร์กโฟลว์สร้างภาพใน **ComfyUI** และเขียนโค้ดให้โมเดล 3B ตัวนี้ช่วยแต่ง Prompt ภาษาอังกฤษส่งไปวาดภาพโดยอัตโนมัติ